# 04. Sterowanie tabelaryczne
## SARSA, Expected SARSA i Q-learning

W poprzednim notebooku ocenialiśmy zadaną politykę. Teraz uczymy się tablicy $Q(s,a)$, żeby po każdym doświadczeniu aktualizować ocenę akcji i stopniowo zmieniać sposób wyboru ruchów.

**Cel ćwiczenia.** Z jednego przejścia oblicz trzy cele aktualizacji. Następnie sprawdź, jakie trasy powstają, gdy te trzy algorytmy uczą się na planszy z urwiskiem.

Uruchamiaj komórki od góry. Plik **srodowiska.py** powinien leżeć obok notebooka. Wystarczą Python 3 i Jupyter. Po uzupełnieniu każdej funkcji uruchom komórkę z jej testem.


**Wersja studencka.** Uzupełnij trzy funkcje oznaczone komentarzem `TODO`. Dopiero potem uruchom eksperyment. Na końcu przygotuj krótki raport oparty na otrzymanych liczbach.


## 1. Jak agent wybiera akcję?

W stanie $s$ polityka $\varepsilon$-zachłanna losuje wśród wszystkich akcji z prawdopodobieństwem $\varepsilon$. Z pozostałą szansą dzieli wybór równo między akcje o największej wartości $Q(s,a)$. To ta sama reguła, której używaliśmy przy bandytach, teraz zapisywana osobno dla każdego stanu.

Gdy masz $m$ akcji, każda dostaje udział $\varepsilon/m$. Gdy $k$ akcji remisuje na pierwszym miejscu, każda z nich dostaje dodatkowo $(1-\varepsilon)/k$. Dzięki temu nawet akcja z mniejszą oceną może dostarczyć doświadczenia.

**Ćwiczenie 1.** Dla czterech ocen $[4,1,2,0]$ i $\varepsilon=0{,}2$ wyprowadź cztery prawdopodobieństwa. Powtórz dla remisu $[4,1,4,0]$, zanim uruchomisz test.


In [ ]:
import math
import random
from statistics import mean, stdev

from srodowiska import CliffWalking


In [ ]:
def epsilon_policy(values, epsilon):
    # TODO: policz największą wartość i liczbę akcji, które ją osiągają.
    # Każdej przydziel udział eksploracyjny. Najlepszym dolicz
    # równy udział z pozostałej masy. Zwróć listę w kolejności akcji.
    raise NotImplementedError("Uzupełnij wybór ε-zachłanny.")


In [ ]:
def close_list(actual, expected):
    return len(actual) == len(expected) and all(
        math.isclose(a, b, abs_tol=1e-12)
        for a, b in zip(actual, expected)
    )

assert close_list(epsilon_policy([4, 1, 2, 0], 0.2), [0.85, 0.05, 0.05, 0.05])
assert close_list(epsilon_policy([4, 1, 4, 0], 0.2), [0.45, 0.05, 0.45, 0.05])
assert close_list(epsilon_policy([4, 1, 4, 0], 0), [0.5, 0, 0.5, 0])
assert close_list(epsilon_policy([4, 1, 2, 0], 1), [0.25] * 4)
print("Reguła wyboru przeszła cztery testy.")


## 2. Jedno doświadczenie, trzy cele

Zaczynamy od przejścia $(s,a,r,s')$. Przez $a'$ oznaczamy komendę, którą polityka zachowania faktycznie wybrała w $s'$. Polityka zachowania steruje zbieraniem doświadczeń. Polityka docelowa mówi, jakie dalsze wybory wyceniamy.

| Algorytm | Cel $y$ przy kontynuacji | Skąd pochodzi dalsza wartość? |
|---|---|---|
| **SARSA** | $r+\gamma Q(s',a')$ | Z jednej komendy wylosowanej z polityki zachowania. |
| **Expected SARSA** | $r+\gamma\sum_b\pi(b\mid s')Q(s',b)$ | Ze średniej po komendach. Tutaj $\pi$ jest polityką zachowania. |
| **Q-learning** | $r+\gamma\max_b Q(s',b)$ | Z najlepiej ocenianej komendy, nawet gdy zachowanie eksploruje. |

Każda metoda stosuje $Q(s,a)\gets Q(s,a)+\alpha[y-Q(s,a)]$, gdzie $\alpha$ to krok uczenia. SARSA i użyta tu wersja Expected SARSA uczą się zgodnie z polityką zachowania (*on-policy*). W Q-learningu dane zbieramy z eksploracją, a oceniamy przyszły wybór zachłanny (*off-policy*).

Zapis SARSA przypomina o kolejności $S,A,R,S',A'$. Gdy epizod się kończy, cel we wszystkich metodach jest równy $r$.

**Ćwiczenie 2.** Dla $r=-1,\ \gamma=0{,}9,\ \varepsilon=0{,}2,\ Q(s',\cdot)=[4,1,2,0]$ przyjmij, że wylosowana komenda ma indeks 2. Policz trzy cele i sprawdź, o ile zmieni się $Q(s,a)=0{,}5$ przy $\alpha=0{,}1$. Następnie potraktuj to samo przejście jako kończące epizod.


In [ ]:
METHODS = ("SARSA", "Expected SARSA", "Q-learning")


def td_target(method, reward, next_values, terminated, gamma, epsilon, next_action=None):
    # TODO: jeśli przejście kończy epizod, zwróć bieżącą nagrodę.
    # W pozostałych przypadkach użyj jednej z trzech reguł
    # dla dalszej wartości. Na końcu dolicz ją z wagą gamma.
    # Dla Expected SARSA wykorzystaj epsilon_policy z zadania 1.
    raise NotImplementedError("Uzupełnij trzy cele aktualizacji.")


def update_q(q, state, action, target, alpha):
    # TODO: oblicz różnicę celu i starej oceny. Przesuń ocenę
    # o alpha tej różnicy. Zmodyfikuj tylko komórkę q[state][action].
    raise NotImplementedError("Uzupełnij aktualizację tablicy Q.")


In [ ]:
values = [4, 1, 2, 0]
target_checks = [0.8, 2.195, 2.6]
update_checks = [0.53, 0.6695, 0.71]

for method, expected_target, expected_update in zip(METHODS, target_checks, update_checks):
    y = td_target(method, -1, values, False, 0.9, 0.2, next_action=2)
    q = [[0.5, 7.0], [2.0, 3.0]]
    update_q(q, 0, 0, y, 0.1)
    assert math.isclose(y, expected_target, abs_tol=1e-12), (method, y)
    assert math.isclose(q[0][0], expected_update, abs_tol=1e-12), (method, q[0][0])
    assert q[0][1] == 7.0 and q[1] == [2.0, 3.0]
    assert td_target(method, -1, None, True, 0.9, 0.2) == -1
    print(f"{method:>14}  cel {y:.3f}   nowa ocena {q[0][0]:.4f}")


## 3. Porównanie na tej samej planszy

Agent startuje w lewym dolnym rogu, a cel jest w prawym dolnym rogu. Pola dolnego rzędu między nimi tworzą urwisko. Zwykły ruch, również wejście do celu, daje $-1$. Wejście na urwisko daje $-100$ i przenosi agenta na start w **tym samym epizodzie**. Dopiero wejście do celu kończy epizod. Położenie agenta się zmienia, podobnie jak u robota po ratunku.

**Przewidywanie przed kodem.** Wyobraź sobie, że stoisz tuż nad urwiskiem. Polityka z $\varepsilon=0{,}1$ czasem wykona komendę „w dół”. Który cel TD uwzględni poziom oceny tej komendy w następnym stanie? Który bierze pod uwagę wyłącznie najlepiej ocenianą akcję?

Wszystkie metody uczymy od tablicy zer i przy wspólnych parametrach, ale z oddzielnymi generatorami dla polityki i środowiska. Na planszy deterministycznej losowanie akcji jest źródłem różnic. W kolejnych powtórzeniach używamy niezależnych par generatorów. Porównujemy dwa tryby pomiaru:

- z eksploracją $\varepsilon=0{,}1$, żeby zmierzyć zachowanie w warunkach treningu
- zachłannie, $\varepsilon=0$, żeby sprawdzić decyzje wynikające z ocen po treningu

Limit kroków chroni eksperyment przed wyjątkowo długim epizodem. Po tym limicie kończymy **pomiar** epizodu. Aktualizacja ostatniego ruchu uwzględnia znacznik zakończenia zwrócony przez środowisko.


In [ ]:
def choose_action(values, epsilon, rng):
    probabilities = epsilon_policy(values, epsilon)
    return rng.choices(range(len(values)), weights=probabilities, k=1)[0]


def run_episode(env, q, method, epsilon, alpha, gamma, max_steps, policy_rng, env_rng):
    state = env.start
    action = choose_action(q[state], epsilon, policy_rng)
    total_reward = 0
    falls = 0
    for _ in range(max_steps):
        next_state, reward, terminated = env.sample(state, action, env_rng)
        total_reward += reward
        falls += reward == -100
        next_action = None
        if not terminated and (method == "SARSA" or method is None):
            next_action = choose_action(q[next_state], epsilon, policy_rng)
        if method is not None:
            target = td_target(
                method, reward, q[next_state], terminated, gamma, epsilon, next_action
            )
            update_q(q, state, action, target, alpha)
        if terminated:
            return total_reward, falls, True
        state = next_state
        action = (
            next_action if next_action is not None
            else choose_action(q[state], epsilon, policy_rng)
        )
    return total_reward, falls, False


def generators(seed):
    source = random.Random(seed)
    return random.Random(source.getrandbits(64)), random.Random(source.getrandbits(64))


def train(env, method, seed, episodes, epsilon=0.1, alpha=0.25, gamma=1.0, max_steps=500):
    q = [[0.0] * len(env.actions) for _ in range(env.n_states)]
    policy_rng, env_rng = generators(seed)
    history = [
        run_episode(env, q, method, epsilon, alpha, gamma, max_steps, policy_rng, env_rng)
        for _ in range(episodes)
    ]
    return q, history


def evaluate(env, q, seed, epsilon, repetitions=75, max_steps=500):
    policy_rng, env_rng = generators(seed)
    return [
        run_episode(env, q, None, epsilon, 0, 1, max_steps, policy_rng, env_rng)
        for _ in range(repetitions)
    ]


In [ ]:
env = CliffWalking()
next_state, reward, terminated = env.sample(env.start, 2, random.Random(0))
assert (next_state, reward, terminated) == (env.start, -100, False)
q_small, run_small = train(env, "SARSA", seed=41, episodes=3, max_steps=20)
q_before = [row[:] for row in q_small]
assert len(run_small) == 3
assert len(evaluate(env, q_small, seed=106, epsilon=0.1, repetitions=2, max_steps=20)) == 2
assert q_small == q_before, "Ocena nie powinna zmieniać tablicy Q."
print("Test środowiska, krótkiego treningu i zamrożonej oceny zakończony.")


In [ ]:
SEEDS = (7, 31, 81, 116, 211, 413)
EPSILON = 0.1
ALPHA = 0.25
EPISODES = 600

trained = {
    (method, seed): train(env, method, seed, EPISODES, epsilon=EPSILON, alpha=ALPHA)
    for method in METHODS for seed in SEEDS
}

print(f"{'metoda':>14} {'tryb':>15} {'średni zwrot':>14} {'SD ziaren':>10} {'spadki/ep.':>12} {'cel [%]':>9}")
for method in METHODS:
    for epsilon in (EPSILON, 0.0):
        samples_by_seed = [
            evaluate(env, trained[method, seed][0], seed + 100_000, epsilon)
            for seed in SEEDS
        ]
        rewards = [mean(row[0] for row in sample) for sample in samples_by_seed]
        falls = [mean(row[1] for row in sample) for sample in samples_by_seed]
        successes = [mean(row[2] for row in sample) for sample in samples_by_seed]
        label = "z eksploracją" if epsilon > 0 else "zachłannie"
        print(f"{method:>14} {label:>15} {mean(rewards):14.2f} {stdev(rewards):10.2f} "
              f"{mean(falls):12.3f} {100 * mean(successes):9.1f}")


**Odczyt tabeli.** Zwrot jest sumą wypłat w epizodzie, łącznie z wydatkami za kroki i upadki. „SD ziaren” to odchylenie standardowe między średnimi z niezależnych ziaren treningu i oceny. „Cel [%]” pokazuje odsetek epizodów zakończonych wejściem do celu w limicie kroków. To pomiar dla wybranych ustawień i zbioru ziaren. W tym zadaniu używamy γ = 1, więc każda nagroda w epizodzie wchodzi do zwrotu z jednakową wagą.


In [ ]:
def greedy_route(env, q, limit=50):
    state = env.start
    visited = [state]
    rng = random.Random(0)
    for _ in range(limit):
        # Przy remisie wybieramy pierwszy indeks, aby wydruk dało się odtworzyć.
        action = max(env.actions, key=q[state].__getitem__)
        state, _, terminated = env.sample(state, action, rng)
        visited.append(state)
        if terminated:
            return visited, True
    return visited, False


DEMO_SEED = SEEDS[2]
for method in METHODS:
    q, _ = trained[method, DEMO_SEED]
    route, success = greedy_route(env, q)
    print(f"\n{method}, ziarno {DEMO_SEED}, dotarcie do celu: {success}")
    marks = ["·"] * env.n_states
    for position in route:
        marks[position] = "×"
    for position in range(37, 47):
        marks[position] = "~"
    marks[env.start], marks[47] = "S", "C"
    for row in range(4):
        print(" ".join(marks[12 * row:12 * (row + 1)]))
print("\n× odwiedzone pole, ~ urwisko, S start, C cel")


## 4. Oddaj krótki raport

1. Zapisz trzy cele dla przejścia z ćwiczenia 2. Wskaż, która metoda korzysta z próbki kolejnej akcji, która z jej wartości oczekiwanej i która z maksimum.
2. Porównaj co najmniej dwie metody przy tym samym trybie oceny. Podaj średni zwrot, liczbę upadków na epizod i odsetek dotarć do celu. Opisz, co zmieniło wyłączenie eksploracji. Wskaż, ile widocznych różnic może wynikać z rozrzutu między ziarnami.
3. Wybierz jedną parę metod i jedno ziarno. Opisz różnicę między trasami, odwołując się do możliwej dalszej komendy przy urwisku. Jeśli trasy się pokrywają, porównaj liczby z oceny z eksploracją.

**Rozszerzenie na kartce.** Sterowanie Monte Carlo aktualizuje $Q(s,a)$ ze zrealizowanego zwrotu po parze $(s,a)$ i stosuje politykę $\varepsilon$-zachłanną względem bieżących ocen. Ułóż kolejność kroków po epizodzie. Wskaż, kiedy zaktualizujesz ocenę, jeśli epizod jeszcze trwa. Zestaw odpowiedź z aktualizacją SARSA po jednym ruchu.

**Przejście do metod wielokrokowych.** Po jednym ruchu możemy uzupełnić brakujący dalszy zwrot oceną $Q$. Po dwóch lub trzech ruchach mamy więcej rzeczywistych nagród. Następny notebook pokazuje pośrednie cele i ślady kwalifikowalności.
